<h2> Query and Performance Analysis</h2>

<h5> Setup

In [1]:
import pandas as pd
import pymysql
import getpass # May need to import as `from getpass import getpass`
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import sqlite3

In [2]:
conn = sqlite3.connect("flights.db")

In [3]:
airlines = pd.read_csv("airlines.csv")
airports = pd.read_csv("airports.csv")
flights = pd.read_csv("flights.csv", low_memory=False)

In [4]:
airlines.to_sql("airlines", conn, if_exists="replace", index=False)
airports.to_sql("airports", conn, if_exists="replace", index=False)
flights.to_sql("flights", conn, if_exists="replace", index=False) #big dataset, takes a while to load

5819079

**Note: shorter wall time = better performance**

<h3> Query 1: Departure Avg Aggregation

**Case 1: No Indexes**

In [5]:
conn.execute("DROP INDEX IF EXISTS idx_airline;")
pd.set_option('display.max_colwidth', None)


In [6]:
%%time
pd.read_sql_query('''
            EXPLAIN QUERY PLAN
            SELECT a.AIRLINE, AVG(f.DEPARTURE_DELAY) AS Average_Delay
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE 
            WHERE DEPARTURE_DELAY > 0 AND DEPARTURE_DELAY IS NOT NULL 
                AND DEPARTURE_TIME IS NOT NULL 
                AND ARRIVAL_TIME IS NOT NULL
            GROUP BY a.AIRLINE
            ORDER BY AVG(f.DEPARTURE_DELAY) ASC
            ;''', conn)

CPU times: user 654 μs, sys: 1.55 ms, total: 2.2 ms
Wall time: 1.84 ms


,id,parent,notused,detail
0,8,0,216,SCAN f
1,31,0,53,SEARCH a USING AUTOMATIC COVERING INDEX (IATA_CODE=?)
2,36,0,0,USE TEMP B-TREE FOR GROUP BY
3,76,0,0,USE TEMP B-TREE FOR ORDER BY


Running the query with no index leads to a wall time of 2.65 ms and the EXPLAIN reveals there's a full table scan on the flights table. This means every record in the flights table is being individually read by the scanner. This is extremely time-intensive and costly since the flights table is 5 million records long. Putting an index here might be appropriate to speed up the JOIN. 

**Case 2: Index on flights(AIRLINES)**

The JOIN clause uses flights(AIRLINES) and airlines(IATA_CODE). To speed up the join, the index should be put on the foreign key, which is flights(AIRLINES). 

In [7]:
conn.execute("DROP INDEX IF EXISTS idx_airline;")
conn.execute("""CREATE INDEX idx_airline
ON flights(AIRLINE);
""")


In [8]:
%%time
pd.read_sql_query('''
            EXPLAIN QUERY PLAN
            SELECT a.AIRLINE, AVG(f.DEPARTURE_DELAY) AS Average_Delay
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE 
            WHERE DEPARTURE_DELAY > 0 AND DEPARTURE_DELAY IS NOT NULL 
                AND DEPARTURE_TIME IS NOT NULL 
                AND ARRIVAL_TIME IS NOT NULL
            GROUP BY a.AIRLINE
            ORDER BY AVG(f.DEPARTURE_DELAY) ASC
            ;''', conn)

CPU times: user 480 μs, sys: 60 μs, total: 540 μs
Wall time: 510 μs


,id,parent,notused,detail
0,9,0,216,SCAN a
1,11,0,61,SEARCH f USING INDEX idx_airline (AIRLINE=?)
2,29,0,0,USE TEMP B-TREE FOR GROUP BY
3,69,0,0,USE TEMP B-TREE FOR ORDER BY


Creating an index on flights(AIRLINE) causes the full table scan to be done on the airlines table instead, which is better for performance reasons since airlines is a much smaller table. Additionally, the wall time significantly reduced down to about 871 microseconds, meaning it was worth the overhead cost of creating an index. It's safe to conclude creating an index on flights(AIRLINE) was appropriate and needed to better performance. 

<h3> Query 2: Delay Analysis Query 

**Case 1: No Index**

In [9]:
conn.execute("DROP INDEX IF EXISTS idx_airline;")

In [10]:
%%time
pd.read_sql_query('''
        EXPLAIN QUERY PLAN
            select
    airline,
    avg(departure_delay) as avg_departure_delay
from flights
where cancelled = 0
group by airline
order by avg_departure_delay desc
            ;''', conn)

CPU times: user 425 μs, sys: 68 μs, total: 493 μs
Wall time: 459 μs


,id,parent,notused,detail
0,7,0,216,SCAN flights
1,11,0,0,USE TEMP B-TREE FOR GROUP BY
2,50,0,0,USE TEMP B-TREE FOR ORDER BY


The wall time here is about 2.23 ms and the EXPLAIN shows the entire flights table is being scanned. Since flights is a large table, putting an index on flights (even though there's no join) might be appropriate. 

**Case 2: Index on flight(AIRLINES)**

In [11]:
conn.execute("DROP INDEX IF EXISTS idx_airline;")
conn.execute("""CREATE INDEX idx_airline
ON flights(AIRLINE);
""")

In [12]:
%%time
pd.read_sql_query('''
        EXPLAIN QUERY PLAN
            select
    airline,
    avg(departure_delay) as avg_departure_delay
from flights
where cancelled = 0
group by airline
order by avg_departure_delay desc
            ;''', conn)

CPU times: user 438 μs, sys: 63 μs, total: 501 μs
Wall time: 432 μs


,id,parent,notused,detail
0,8,0,223,SCAN flights USING INDEX idx_airline
1,42,0,0,USE TEMP B-TREE FOR ORDER BY


The wall time after creating the index is about 1.14 ms, which is about half the wall time from when there was no index. Additionally, instead of scanning every single row in flights, it's scanning using the index, meaning it's more efficient and cost-friendly. Because of this index, there was no longer any need for a temporary B-TREE to be created for the GROUP BY clause like before. Less temporary B-TREE = less memory overhead and minimize disk I/O. Since this index reduced the need for temp B-TREEs and reduced wall time, putting an index here was a really good choice. 

**Case 3: Adding another Index on flights(CANCELLED)**

Another classic choice for indexes are columns involved in the predicates (WHERE clauses). In that case, let's see what happens if we add another index on flights(CANCELLED) since the CANCELLED column is in the WHERE clause. 

In [13]:
conn.execute("DROP INDEX IF EXISTS idx_airline;")
conn.execute("DROP INDEX IF EXISTS idx_cancelled;")
conn.execute("""CREATE INDEX idx_airline
ON flights(AIRLINE);
""")
conn.execute("""CREATE INDEX idx_cancelled
ON flights(CANCELLED);
""")

In [14]:
%%time
pd.read_sql_query('''
        EXPLAIN QUERY PLAN
            select
    airline,
    avg(departure_delay) as avg_departure_delay
from flights
where cancelled = 0
group by airline
order by avg_departure_delay desc
            ;''', conn)

#indexing on a low cardinality column like cancelled is worse for performance. no index is better here

CPU times: user 627 μs, sys: 438 μs, total: 1.06 ms
Wall time: 794 μs


,id,parent,notused,detail
0,8,0,60,SEARCH flights USING INDEX idx_cancelled (CANCELLED=?)
1,13,0,0,USE TEMP B-TREE FOR GROUP BY
2,52,0,0,USE TEMP B-TREE FOR ORDER BY


The wall time here is about 1.86 ms, which is slightly higher than when there was only one index on flights(AIRLINE). The reason why this index performed worse is because the CANCELLED column has low cardinality, or in other words, this column has a small number of distinct values. Thus, an index on flights(CANCELLED) is a poor choice for performance. 

<h3> Query 3: Comparing MySQL and MongoDB query on Avg Arrivals Delay

**MongoDB query**

In [15]:
import pymongo
from pymongo import MongoClient
import pandas as pd
client = MongoClient('localhost', 27017)
db = client.example

In [16]:
flights_airlines = pd.merge(airlines, flights, left_on='IATA_CODE', right_on = 'AIRLINE', how = 'inner' )

In [17]:
sample = flights_airlines.sample(10000, random_state=1)
sample_dict = sample.to_dict('records')
flights_airlines_collection = db.flights_airlines
result0 = flights_airlines_collection.insert_many(sample_dict)

In [24]:
from pprint import pprint
pipeline = [
    {'$match': {'ARRIVAL_DELAY': {'$gt': 0, '$ne': None},
            'ARRIVAL_TIME': {'$ne': None},
            'DIVERTED': 0,
            'CANCELLED': 0}},
    {'$group': {'_id': '$AIRLINE_x', 
                'Average_Arrival_Delay': {'$avg': '$ARRIVAL_DELAY'}}},
    {'$sort': {'Average_Arrival_Delay': 1}}
]



In [25]:
%%time

list(flights_airlines_collection.aggregate(pipeline))

CPU times: user 2.53 ms, sys: 3.6 ms, total: 6.13 ms
Wall time: 4.23 s


[{'_id': 'Hawaiian Airlines Inc.', 'Average_Arrival_Delay': 15.34575041662582},
 {'_id': 'Alaska Airlines Inc.', 'Average_Arrival_Delay': 22.538584178147474},
 {'_id': 'US Airways Inc.', 'Average_Arrival_Delay': 27.393178735223824},
 {'_id': 'Southwest Airlines Co.',
  'Average_Arrival_Delay': 29.407610825309348},
 {'_id': 'Virgin America', 'Average_Arrival_Delay': 30.567634038366947},
 {'_id': 'Delta Air Lines Inc.', 'Average_Arrival_Delay': 32.02885794704069},
 {'_id': 'Skywest Airlines Inc.', 'Average_Arrival_Delay': 32.435623793570144},
 {'_id': 'American Airlines Inc.', 'Average_Arrival_Delay': 34.11261037452026},
 {'_id': 'Atlantic Southeast Airlines',
  'Average_Arrival_Delay': 35.196399269684996},
 {'_id': 'JetBlue Airways', 'Average_Arrival_Delay': 38.19633695378599},
 {'_id': 'United Air Lines Inc.', 'Average_Arrival_Delay': 39.16217848533814},
 {'_id': 'American Eagle Airlines Inc.',
  'Average_Arrival_Delay': 39.45322671343201},
 {'_id': 'Spirit Air Lines', 'Average_Arrival

**MySQLquery**

In [23]:
sample.to_sql("sample", conn, if_exists="replace", index=False)

10000

In [ ]:
%%time
pd.read_sql_query('''
            SELECT AIRLINE_x, AVG(ARRIVAL_DELAY)
            FROM sample
            WHERE ARRIVAL_DELAY > 0 AND ARRIVAL_DELAY IS NOT NULL 
                AND ARRIVAL_TIME IS NOT NULL 
                AND DIVERTED == 0 
                AND CANCELLED == 0
            GROUP BY AIRLINE_x
            ORDER BY AVG(ARRIVAL_DELAY) ASC
            ;''', conn)

CPU times: user 2.49 ms, sys: 1.03 ms, total: 3.52 ms
Wall time: 2.4 ms


,AIRLINE_x,AVG(ARRIVAL_DELAY)
0,Hawaiian Airlines Inc.,12.924528
1,Virgin America,12.925926
2,Alaska Airlines Inc.,20.734043
3,US Airways Inc.,25.571429
4,Delta Air Lines Inc.,28.591422
5,Southwest Airlines Co.,28.612903
6,American Airlines Inc.,31.473068
7,Skywest Airlines Inc.,32.309589
8,Atlantic Southeast Airlines,35.084399
9,American Eagle Airlines Inc.,35.361582


The MongoDB's query had a wall time of about 4.23 ms whereas the MySQL's query had a wall time of about 2.4 ms. In most instances, MySQL would perform better than MongoDB because MySQL is better suited for aggregations and joins than MongoDB is. The denormalized nature of MongoDB makes it highly inefficient (and counterintuitive when the data is already structured) to conduct joins and aggregations, especially without an index. In short, the MySQL database is a better choice due to the vast size, the normalized structure, and the joins and aggregation between large tables.